In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Input
import os
from tensorflow.keras.applications import DenseNet121



2025-09-10 09:21:39.785159: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-10 09:21:39.795714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757488899.807288  749991 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757488899.810613  749991 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-10 09:21:39.822025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:

# ----------------------------
# Config
# ----------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
DROPOUT_RATE = 0.8
N_CLASSES = 3   # Normal, Benign, Malignant


In [3]:

# ----------------------------
# Data pipeline
# ----------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    "CDD-CESM/organized_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    color_mode="rgb"  # important : force 3 canaux
)

val_generator = train_datagen.flow_from_directory(
    "CDD-CESM/organized_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    color_mode="rgb"
)

batch_x, batch_y = next(train_generator)
print("Sample batch shape:", batch_x.shape)  # doit être (16,224,224,3)


Found 1601 images belonging to 3 classes.
Found 400 images belonging to 3 classes.
Sample batch shape: (16, 224, 224, 3)


In [5]:

# ----------------------------
# Model definition (patch skip_mismatch)
# ----------------------------
inputs = Input(shape=IMG_SIZE + (3,))
base_model = DenseNet121(weights=None, include_top=False, input_tensor=inputs)

# Télécharger les poids officiels
weights_path = tf.keras.utils.get_file(
    'densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5',
    'https://storage.googleapis.com/tensorflow/keras-applications/densenet/densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5',
    cache_subdir='models'
)

# Charger les poids en ignorant les couches incompatibles (stem_conv)
base_model.load_weights(weights_path, by_name=True, skip_mismatch=True)
print("DenseNet121 loaded with skip_mismatch=True")

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(DROPOUT_RATE)(x)  # large dropout (0.8) as per paper
output = Dense(N_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# Fine-tune ALL layers
for layer in base_model.layers:
    layer.trainable = True


DenseNet121 loaded with skip_mismatch=True


In [6]:

# ----------------------------
# Compile
# ----------------------------
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


In [7]:

# ----------------------------
# Train
# ----------------------------
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)


/home/light/miniforge3/envs/pytorch/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 542s 5s/step - accuracy: 0.3448 - loss: 1.7209 - precision: 0.3466 - recall: 0.2780 - val_accuracy: 0.3300 - val_loss: 2395.6907 - val_precision: 0.3300 - val_recall: 0.3300 - learning_rate: 0.0010
Epoch 2/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 449s 4s/step - accuracy: 0.3766 - loss: 1.4339 - precision: 0.4021 - recall: 0.2436 - val_accuracy: 0.3300 - val_loss: 4.2727 - val_precision: 0.3300 - val_recall: 0.3300 - learning_rate: 0.0010
Epoch 3/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 445s 4s/step - accuracy: 0.3716 - loss: 1.3704 - precision: 0.3692 - recall: 0.2080 - val_accuracy: 0.4150 - val_loss: 1.2072 - val_precision: 0.3788 - val_recall: 0.1875 - learning_rate: 0.0010
Epoch 4/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 380s 4s/step - accuracy: 0.3923 - loss: 1.2855 - precision: 0.3905 - recall: 0.1993 - val_accuracy: 0.3350 - val_loss: 31.8782 - val_precision: 0.3455 - val_recall: 0.3325 - learning_rate: 0.0010
Epoch 5/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 446s 4s/step - a

In [8]:

# ----------------------------
# Evaluate
# ----------------------------
loss, acc, prec, rec = model.evaluate(val_generator)
print(f"Validation Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}, Recall: {rec:.4f}")


25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 798ms/step - accuracy: 0.5425 - loss: 0.9839 - precision: 0.6113 - recall: 0.3775
Validation Accuracy: 0.5425
Precision: 0.6113, Recall: 0.3775


In [9]:
from sklearn.metrics import confusion_matrix

# Get true labels and predictions from the validation generator
import numpy as np

val_steps = val_generator.samples // val_generator.batch_size
y_true = []
y_pred = []

for i in range(val_steps):
	x_batch, y_batch = next(val_generator)
	y_true.extend(np.argmax(y_batch, axis=1))
	preds = model.predict(x_batch)
	y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
print(cm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 754ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 585ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 667ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 680ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 615ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 565ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 627ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 575ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 600ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 757ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 745ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 815ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 718ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 800ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 755ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 744ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 699ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 794ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 750ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 729ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 770ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 765ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 897ms/step
[[  6  27  84]
 [  9  76  47

In [11]:
#Save model

model.save("CDD_DenseNet121_model_1.h5")
